# 05 - Optimizacion promocional

## 1. Objetivo
**Pregunta:** que descuento y duracion balancean retorno, crecimiento y riesgo? **Decision:** simular tres casos y separar optimo matematico, robusto y crecimiento. **Tecnico:** se usan champions congelados. **Negocio:** maximizar ROI puntual no basta. **Resultado:** recomendaciones trazables por escenario. **Limitacion:** no es evidencia causal.

## 2. Contexto de negocio
**Pregunta:** que aprobaria Trade Marketing? **Decision:** exigir ROI lower 90 positivo y soporte. **Tecnico:** incertidumbre OOF y guardrails. **Negocio:** reduce promociones destructivas. **Resultado:** existe salida `NO_RECOMMENDATION`. **Limitacion:** faltan margen, stock y presupuesto.

## 3. Trazabilidad del codigo
**Pregunta:** como reproducir la decision? **Decision:** una llamada al pipeline. **Tecnico:** logica en `src/mini_tpo/`. **Negocio:** evita diferencias entre notebook y artifact. **Resultado:** tablas, figuras y modelos se regeneran juntos. **Limitacion:** el registry registra una nueva fecha de refit.


In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').exists(): raise FileNotFoundError('No se encontro la raiz del proyecto')
sys.path.insert(0, str(ROOT / 'src'))
from mini_tpo.pipeline import run_optimization
outputs = run_optimization()  # unica llamada
TABLES = ROOT / 'reports' / 'tables'
def table(name): return pd.read_csv(TABLES / name)
print('Fecha de referencia:', outputs['reference_date'].date())
print('Escenarios:', len(outputs['scenarios']), '| q90 uplift:', round(outputs['q90_uplift'],4), '| q90 ROI:', round(outputs['q90_roi'],4))


## 4. Modelos utilizados
**Pregunta:** que pipelines optimizan? **Decision:** production refits de champions Ridge y boosting ROI directo. **Tecnico:** mismos features e hiperparametros, 2,048 filas, sin tuning. **Negocio:** aprovecha toda la historia tras cerrar evaluacion. **Resultado:** dos joblib nuevos. **Limitacion:** artifacts dependen de version sklearn.

## 5. Decisiones heredadas
**Pregunta:** puede cambiar el champion? **Decision:** no. **Tecnico:** test ya fue abierto en fase 04. **Negocio:** evita escoger por resultados convenientes. **Resultado:** challengers solo miden sensibilidad. **Limitacion:** una futura recalibracion requerira protocolo nuevo.

## 6. Seleccion de tres SKUs
**Pregunta:** que riesgos deben representarse? **Decision:** soporte alto, alto volumen y SKU reciente. **Tecnico:** ranking reproducible de soporte/volumen/fecha. **Negocio:** no se muestran solo casos faciles. **Resultado:** PD008, PD013 y PD015 en cadenas distintas. **Limitacion:** son casos ilustrativos, no portafolio completo.

## 7. Contexto de prediccion
**Pregunta:** que snapshot futuro usar? **Decision:** 29-12-2025, mediana de ultimas cuatro observaciones y promocion no secundaria. **Tecnico:** regla robusta y reproducible. **Negocio:** reduce ruido reciente. **Resultado:** contexto por SKU-cadena. **Limitacion:** no existe forecast externo de baseline.


In [ ]:
display(table('selected_optimization_cases.csv'))
display(table('optimization_contexts.csv'))
registry = json.loads((ROOT/'models/model_registry.json').read_text(encoding='utf-8'))
display(pd.DataFrame([m for m in registry['models'] if m['role']=='production_refit'])[['target','family','feature_set','artifact','train_period']])


## 8. Espacio de busqueda
**Pregunta:** que combinaciones evaluar? **Decision:** 5%-40% cada 1 pp y 5/7/10/14/21 dias. **Tecnico:** producto cartesiano determinista. **Negocio:** resolucion accionable sin falsa precision. **Resultado:** 180 escenarios por caso. **Limitacion:** enteros intermedios son solo sensibilidad.

## 9. Recalculo de features
**Pregunta:** como evitar inconsistencia train-simulacion? **Decision:** reutilizar `build_engineered_core`. **Tecnico:** recalcula interacciones, cuadrados, volumen y calendario. **Negocio:** cada palanca representa el escenario real. **Resultado:** esquema coincide con champions. **Limitacion:** no modela inversion no observada futura.

## 10. Prediccion de uplift
**Pregunta:** cuanto aumenta relativamente la demanda? **Decision:** predecir sin clipping silencioso. **Tecnico:** flags para valores invalidos. **Negocio:** uplift no equivale a unidades ni rentabilidad. **Resultado:** uplift y bandas 90%. **Limitacion:** asociacion predictiva, no causal.

## 11. Curva de demanda
**Pregunta:** cuantas unidades representa el uplift? **Decision:** calcular baseline de tanda, incremento y volumen promocional. **Tecnico:** volumen base por duracion. **Negocio:** hace visible el impacto absoluto. **Resultado:** curvas con bandas en unidades. **Limitacion:** no incorpora stock-out.

## 12. Prediccion de ROI
**Pregunta:** que retorno anticipa cada escenario? **Decision:** usar champion ROI directo. **Tecnico:** nunca usa uplift real. **Negocio:** evita maximizar solo volumen. **Resultado:** ROI esperado por grilla. **Limitacion:** formula contable oficial no esta disponible.


In [ ]:
display(table('optimization_grid_summary.csv'))
scenarios = pd.read_parquet(ROOT/'data/processed/optimization_scenarios.parquet')
display(scenarios[['case_id','factor_descuento','duracion_dias','uplift_esperado','volumen_promo_esperado','volumen_incremental_esperado','roi_esperado','local_support_count','flag_extrapolacion']].head())


## 13. Incertidumbre
**Pregunta:** cuanto error acompana cada punto? **Decision:** q90 absoluto OOF de champions. **Tecnico:** no usa test ni residuos in-sample. **Negocio:** penaliza decisiones fragiles. **Resultado:** bandas empiricas constantes mas penalizacion por soporte. **Limitacion:** no es conformal independiente.

## 14. Soporte global y local
**Pregunta:** existe evidencia cerca del candidato? **Decision:** misma duracion y descuento +/-2.5 pp. **Tecnico:** distancia, rango y conteo local. **Negocio:** estar dentro del dominio global no basta. **Resultado:** flags de extrapolacion por escenario. **Limitacion:** conteos bajos siguen teniendo alta varianza.

## 15. Guardrails
**Pregunta:** cuando puede recomendarse automaticamente? **Decision:** ROI lower 90 positivo, soporte no insuficiente, soporte local, duracion observada y sin desacuerdo material. **Tecnico:** filtro booleano auditable. **Negocio:** reduce aprobaciones destructivas. **Resultado:** candidatos factibles por caso. **Limitacion:** faltan restricciones de presupuesto/stock.

## 16. Optimo matematico
**Pregunta:** donde es maximo el ROI puntual? **Decision:** maximizar sin ocultar limites o extrapolacion. **Tecnico:** argmax determinista. **Negocio:** cumple objetivo literal y muestra su riesgo. **Resultado:** suele favorecer descuentos minimos sin soporte local. **Limitacion:** no es la recomendacion principal.

## 17. Optimo robusto
**Pregunta:** cual es la mejor decision defendible? **Decision:** maximizar ROI robusto dentro de guardrails. **Tecnico:** empate dentro de epsilon favorece volumen, soporte y menor complejidad. **Negocio:** balancea retorno y evidencia. **Resultado:** recomendacion principal o NO_RECOMMENDATION. **Limitacion:** penalizaciones son politica transparente, no precision estadistica.

## 18. Crecimiento rentable
**Pregunta:** cuanto volumen adicional puede capturarse conservando rentabilidad? **Decision:** maximizar unidades entre escenarios robustos. **Tecnico:** misma factibilidad, objetivo distinto. **Negocio:** muestra costo de oportunidad de priorizar ROI. **Resultado:** alternativa por caso. **Limitacion:** no incluye capacidad ni canibalizacion.

## 19. Pareto
**Pregunta:** que escenarios no son dominados en ROI y volumen? **Decision:** construir frontera por caso. **Tecnico:** dominancia bivariada exacta. **Negocio:** facilita negociar retorno versus crecimiento. **Resultado:** tabla y grafico. **Limitacion:** no agrega inversion como tercer eje.


In [ ]:
display(table('mathematical_roi_optima.csv')[['case_id','factor_descuento','duracion_dias','roi_esperado','local_support_count','flag_extrapolacion']])
display(table('optimal_recommendations.csv'))
display(table('profitable_growth_alternatives.csv')[['case_id','factor_descuento','duracion_dias','volumen_incremental_esperado','roi_lower_90']])
display(table('optimization_pareto_frontier.csv').groupby('case_id').size().rename('escenarios_pareto'))


## 20. Caso A
**Pregunta:** que recomienda el caso de mayor soporte? **Decision:** usar optimo robusto PD008-Cadena01. **Tecnico:** soporte alto y evidencia local. **Negocio:** referencia de confianza relativa. **Resultado:** 9% por 14 dias. **Limitacion:** sensibilidad de contexto cambia la decision y exige seguimiento.

## 21. Caso B
**Pregunta:** como pesa el error en unidades? **Decision:** optimizar PD013-Cadena02 y comparar crecimiento. **Tecnico:** volumen base alto. **Negocio:** pequenos cambios de uplift movilizan muchas unidades. **Resultado:** 10% por 14 dias. **Limitacion:** soporte medio, no alto.

## 22. Caso C
**Pregunta:** puede automatizarse un SKU reciente? **Decision:** mostrar candidato pero exigir revision humana. **Tecnico:** soporte bajo y mayor extrapolacion. **Negocio:** evita falsa confianza. **Resultado:** 12% por 21 dias. **Limitacion:** solo 15 observaciones historicas.

## 23. Comparacion de recomendaciones
**Pregunta:** que cambia entre casos? **Decision:** comparar ROI, unidades, intervalos y soporte. **Tecnico:** mismas definiciones. **Negocio:** prioriza segun objetivo comercial. **Resultado:** tabla unica trazable. **Limitacion:** ROI no se traduce a margen absoluto.

## 24. Sensibilidad
**Pregunta:** la recomendacion resiste supuestos alternativos? **Decision:** challengers, ultimo valor, secundaria, duraciones enteras e intervalo 80%. **Tecnico:** no cambia champions. **Negocio:** identifica decisiones que requieren cautela. **Resultado:** PD013 es mas estable; PD008 cambia con contexto. **Limitacion:** no cubre shocks externos.

## 25. Valor de negocio
**Pregunta:** que aporta el Mini-TPO? **Decision:** convertir predicciones en opciones con riesgo visible. **Tecnico:** escenarios, soporte y Pareto. **Negocio:** reduce decisiones basadas solo en uplift o intuicion. **Resultado:** recomendacion, alternativa y razones. **Limitacion:** sigue requiriendo juicio comercial.

## 26. Limitaciones
**Pregunta:** que no debe inferirse? **Decision:** no afirmar causalidad ni garantia. **Tecnico:** datos observacionales y error OOF. **Negocio:** faltan margen, costos, stock, presupuesto y canibalizacion. **Resultado:** advertencias explicitas. **Limitacion:** la precision futura puede degradarse.

## 27. Proximos pasos
**Pregunta:** que falta antes de produccion? **Decision:** incorporar restricciones economicas, experimentos y monitoreo. **Tecnico:** versionado, drift y recalibracion. **Negocio:** gobernanza de aprobacion humana. **Resultado:** roadmap claro. **Limitacion:** fuera del alcance de esta prueba reducida.


In [ ]:
display(table('optimization_sensitivity.csv'))
for case in table('selected_optimization_cases.csv')['case_id']:
    for plot in ['demand','uplift','roi','heatmap','pareto']:
        display(Image(filename=ROOT/'reports/figures/optimization'/f'{case}_{plot}.png'))
required = list(outputs['paths'].values())
assert not [str(p) for p in required if not Path(p).exists()]
assert table('selected_optimization_cases.csv')['id_material'].nunique() == 3
print('Artifacts validados:', len(required), '| Optimizacion ejecutada sin usar targets futuros.')
